# NFL pick'em odds

Scrapes the Las Vegas odds table from vegasinsider.com and averages the point spread across books.
Negative spread = favored (the more negative, the bigger the expected win).

The site's HTML is flaky — columns get scrambled, junk like `--4.5` shows up, and the spread /
total / moneyline sections are all stacked into one table — so the parsing below validates every
cell and silently drops whatever doesn't make sense instead of crashing.

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', None)

In [2]:
url = "https://www.vegasinsider.com/nfl/odds/las-vegas/"
tab = pd.read_html(url)[0]

In [3]:
def parse_line(cell):
    """Pull the leading number out of a cell like '+3.5 -110 +'.
    NaN for anything malformed (e.g. the '--4.5' junk the site sometimes renders)."""
    if not isinstance(cell, str):
        return np.nan
    tok = cell.split()[0]
    if tok.upper() in ('PK', 'PICK', 'EVEN'):
        return 0.0
    if re.fullmatch(r'[+-]?\d+(?:\.\d+)?', tok):
        return float(tok)
    return np.nan


def parse_ml(cell):
    """Moneyline: 'EVEN' / 'EV' mean +100 (parse_line would give 0, which is a spread
    convention and poisons the average). Anything with |odds| < 100 is a bad render."""
    if not isinstance(cell, str):
        return np.nan
    tok = cell.split()[0].upper()
    if tok in ('EVEN', 'EV', 'PK', 'PICK'):
        return 100.0
    v = parse_line(cell)
    return v if abs(v) >= 100 else np.nan


def implied_prob(m):
    """American odds -> implied probability (vig included)."""
    return 100 / (m + 100) if m > 0 else -m / (-m + 100)


def parse_total(cell):
    """Pull the total out of an over/under cell like 'o47.5 -110 +' or 'u 47.5 -105'."""
    if not isinstance(cell, str):
        return np.nan
    m = re.match(r'^[ou]\s*(\d+(?:\.\d+)?)', cell.strip(), re.IGNORECASE)
    return float(m.group(1)) if m else np.nan


def load_sections(tab):
    """The page stacks the spread / total / moneyline sections into one table, with
    rotation numbers restarting at each section. Split them apart, keeping raw cell
    strings (each section parses its own format). Book columns are auto-detected,
    so no more hand-maintaining the sources list."""
    sources = [c for c in tab.columns
               if c not in ('Time', 'Open') and not str(c).startswith('Unnamed')]
    rows, seen, section = [], set(), 0
    for _, r in tab.iterrows():
        t = r['Time']
        if not isinstance(t, str):
            continue
        m = re.match(r'^(\d+)\s+(.*\S)', t)      # team rows look like '451 Patriots'
        if not m:
            continue                              # skips 'Matchup', 'Final', header junk
        rot = int(m.group(1))
        if rot in seen:                           # rotation number repeated -> new section
            section += 1
            seen = set()
        seen.add(rot)
        rows.append({'section': section, 'Team': m.group(2),
                     **{s: r[s] for s in sources}})
    return pd.DataFrame(rows), sources



def drop_swapped(df, sources, mirror, min_consensus, label='lines'):
    """The site sometimes renders a book's column with the two teams swapped
    (confirmed against the book's own site for HardRock). A swapped value sits near
    the *mirror image* of the consensus, so flag a value only if it is closer to
    mirror(median) than to the median AND the consensus is far enough from a toss-up
    that the two are distinguishable. Books genuinely disagreeing about who is
    favored in a pick'em game are kept: that disagreement is real information.

    mirror: function giving the other side's value (-x for spreads, 1-p for probs)
    min_consensus: |median - mirror(median)| must exceed this to flag anything
    """
    df = df.reset_index(drop=True).copy()
    med = df[sources].median(axis=1)
    decidable = (med - mirror(med)).abs() >= min_consensus
    for s in sources:
        v = df[s]
        swapped = decidable & ((v - mirror(med)).abs() < (v - med).abs())
        if swapped.any():
            print(f"{s}: ignoring swapped {label} for {df.loc[swapped, 'Team'].tolist()}")
        df.loc[swapped, s] = np.nan
    return df


def report_splits(df, sources):
    """Print games where the surviving books disagree on who is favored."""
    for g in range(0, len(df) - 1, 2):
        row = df.loc[g, sources]
        neg = [s for s in sources if row[s] < 0]
        pos = [s for s in sources if row[s] > 0]
        if neg and pos:
            print(f"books split: {df.loc[g, 'Team']} favored by {neg} | "
                  f"{df.loc[g + 1, 'Team']} favored by {pos}")


def clean_spreads(df, sources):
    """Games are consecutive row pairs, and a book's two lines in a game must be
    mirror images (+3.5 / -3.5). Anything else is a bad render -> NaN both sides."""
    df = df.reset_index(drop=True).copy()
    for g in range(0, len(df) - 1, 2):
        for s in sources:
            a, b = df.loc[g, s], df.loc[g + 1, s]
            if np.isnan(a) or np.isnan(b) or a != -b:
                df.loc[[g, g + 1], s] = np.nan
    # a swapped column reads e.g. +3.5 where consensus is -3.5; only decidable when the
    # consensus is >= 2.5 points from pick'em (so |med - (-med)| >= 5)
    df = drop_swapped(df, sources, mirror=lambda x: -x, min_consensus=5.0, label='spreads')
    df = df.dropna(subset=sources, how='all')      # drops games already final
    report_splits(df.reset_index(drop=True), sources)
    return df

In [4]:
raw, sources = load_sections(tab)
full = raw.copy()
for s in sources:
    full[s] = raw[s].map(parse_line)

# classify sections by typical magnitude: spreads are small, moneylines are +-100 and up,
# and totals ('o47.5 ...') never parse with parse_line at all
med_abs = full.groupby('section')[sources].apply(lambda d: d.abs().median().median())
spread_sections = med_abs[med_abs < 50].index
ml_sections = med_abs[med_abs >= 100].index
total_sections = med_abs[med_abs.isna()].index

spreads = clean_spreads(full[full['section'].isin(spread_sections)], sources)
spreads['ave_spread'] = spreads[sources].mean(axis=1)
spreads['n_books'] = spreads[sources].notna().sum(axis=1)
spreads[['Team'] + sources + ['ave_spread', 'n_books']].sort_values('ave_spread')

HardRock: ignoring swapped spreads for ['Buccaneers', 'Bengals']
books split: Bills favored by ['Bet365', 'BetMGM', 'DraftKings', 'Caesars', 'FanDuel', 'HardRock', 'RiversCasino', 'Consensus'] | Texans favored by ['Fanatics']
books split: Packers favored by ['BetMGM', 'HardRock'] | Vikings favored by ['Bet365', 'DraftKings', 'Caesars', 'FanDuel', 'Fanatics', 'RiversCasino', 'Consensus']


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_spread,n_books
21,Chargers,-10.5,-10.5,-10.5,-10.5,-9.5,-10.5,-10.0,-10.0,-10.5,-10.277778,9
5,Jaguars,-7.5,-7.5,-7.5,-7.5,-7.5,-8.5,-8.5,-8.0,-7.5,-7.777778,9
19,Lions,-7.0,-7.0,-7.0,-7.0,-6.5,-6.5,-7.0,-7.0,-7.0,-6.888889,9
25,Eagles,-5.5,-4.5,-5.5,-4.5,-5.5,NaN,-5.0,-5.5,-5.5,-5.187500,8
3,Rams,-3.5,-3.5,-3.5,-3.5,-3.5,-4.5,-3.5,-3.5,-3.5,-3.611111,9
7,Bengals,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-4.0,-3.5,-3.5,-3.562500,8
1,Seahawks,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-3.5,-3.5,-3.5,-3.500000,8
23,Raiders,-3.5,-3.5,-3.5,-3.5,-3.5,NaN,-3.5,-3.5,-3.5,-3.500000,8
8,Ravens,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.5,-3.500000,9
11,Steelers,-3.5,-3.0,-3.5,-3.0,-3.0,NaN,-3.5,-3.0,-3.5,-3.250000,8


## Moneyline → implied win probability

The moneyline section of the same table is literally the odds a team wins the matchup.
Each book's line is converted to an implied probability first (American odds are
nonlinear and asymmetric around ±100, so averaging them directly is biased), then the
probabilities are averaged across books and the vig removed by normalizing each game's
two probabilities to sum to 1. Swapped-team columns are dropped; genuine book
disagreement on toss-ups is kept.

In [5]:
ml = full[full['section'].isin(ml_sections)].reset_index(drop=True).copy()
raw_ml = raw[raw['section'].isin(ml_sections)].reset_index(drop=True)
for s in sources:
    ml[s] = raw_ml[s].map(parse_ml).map(lambda m: implied_prob(m) if not np.isnan(m) else np.nan)

# a swapped column reads p where consensus is 1-p; only decidable when consensus is
# >= 5 points from even (so |med - (1-med)| >= 0.10)
ml = drop_swapped(ml, sources, mirror=lambda p: 1 - p, min_consensus=0.10, label='moneylines')
ml = ml.dropna(subset=sources, how='all').reset_index(drop=True)
ml['q'] = ml[sources].mean(axis=1)                # vig-included average implied prob
ml['n_books'] = ml[sources].notna().sum(axis=1)
for g in range(0, len(ml) - 1, 2):
    tot = ml.loc[g, 'q'] + ml.loc[g + 1, 'q']
    ml.loc[[g, g + 1], 'win_prob'] = ml.loc[[g, g + 1], 'q'] / tot
ml[['Team'] + sources + ['q', 'win_prob', 'n_books']].round(3).sort_values('win_prob', ascending=False)

HardRock: ignoring swapped moneylines for ['Rams', 'Jaguars', 'Falcons', 'Saints', 'Giants']


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,q,win_prob,n_books
21,Chargers,0.846,0.867,0.846,0.855,0.855,0.852,0.846,0.848,0.846,0.851,0.814,9
5,Jaguars,0.796,0.810,0.798,0.796,0.800,NaN,0.800,0.794,0.798,0.799,0.767,8
19,Lions,0.759,0.778,0.753,0.765,0.770,0.765,0.767,0.775,0.753,0.765,0.733,9
25,Eagles,0.697,0.688,0.686,0.692,0.706,0.688,0.701,0.697,0.686,0.693,0.665,9
7,Bengals,0.672,0.667,0.658,0.672,0.672,0.655,0.683,0.672,0.658,0.668,0.641,9
3,Rams,0.667,0.667,0.664,0.663,0.664,NaN,0.667,0.672,0.664,0.666,0.639,8
1,Seahawks,0.667,0.667,0.636,0.655,0.664,0.655,0.655,0.667,0.636,0.656,0.628,9
23,Raiders,0.655,0.655,0.664,0.663,0.638,0.649,0.655,0.661,0.664,0.656,0.628,9
8,Ravens,0.643,0.667,0.636,0.645,0.640,0.649,0.643,0.655,0.636,0.646,0.619,9
11,Steelers,0.643,0.615,0.649,0.630,0.638,0.643,0.643,0.639,0.649,0.639,0.612,9


## Picks by matchup

Same numbers, but one line per game in the order the site lists them, so it's quick to
walk down the pick'em sheet. Format: `away (spread, win prob) @ home (spread, win prob) -> pick`.

In [6]:
sp = spreads.set_index('Team')['ave_spread']
wp = ml.set_index('Team')['win_prob']

games = full[full['section'].isin(spread_sections)].reset_index(drop=True)
for g in range(0, len(games) - 1, 2):
    away, home = games.loc[g, 'Team'], games.loc[g + 1, 'Team']
    pa, ph = wp.get(away, np.nan), wp.get(home, np.nan)
    if np.isnan(pa) and np.isnan(ph):
        continue                                  # game already final / no data
    pick = home if (ph if not np.isnan(ph) else -1) >= (pa if not np.isnan(pa) else -1) else away
    print(f"{away:>12} ({sp.get(away, np.nan):+5.1f}, {pa:4.0%})  @  "
          f"{home:<12} ({sp.get(home, np.nan):+5.1f}, {ph:4.0%})   ->  {pick}")

    Patriots ( +3.5,  37%)  @  Seahawks     ( -3.5,  63%)   ->  Seahawks
       49ers ( +3.6,  36%)  @  Rams         ( -3.6,  64%)   ->  Rams
      Browns ( +7.8,  23%)  @  Jaguars      ( -7.8,  77%)   ->  Jaguars
  Buccaneers ( +3.6,  36%)  @  Bengals      ( -3.6,  64%)   ->  Bengals
      Ravens ( -3.5,  62%)  @  Colts        ( +3.5,  38%)   ->  Ravens
     Falcons ( +3.2,  39%)  @  Steelers     ( -3.2,  61%)   ->  Steelers
       Bills ( -1.2,  51%)  @  Texans       ( +1.2,  49%)   ->  Bills
       Bears ( -2.6,  58%)  @  Panthers     ( +2.6,  42%)   ->  Bears
        Jets ( +1.9,  45%)  @  Titans       ( -1.9,  55%)   ->  Titans
      Saints ( +6.9,  27%)  @  Lions        ( -6.9,  73%)   ->  Lions
   Cardinals (+10.3,  19%)  @  Chargers     (-10.3,  81%)   ->  Chargers
    Dolphins ( +3.5,  37%)  @  Raiders      ( -3.5,  63%)   ->  Raiders
  Commanders ( +5.2,  33%)  @  Eagles       ( -5.2,  67%)   ->  Eagles
     Packers ( +0.7,  48%)  @  Vikings      ( -0.7,  52%)   ->  Vikings
 

## Tiebreaker

Over/under total for the last matchup of the week, averaged across books.

In [7]:
tot = raw[raw['section'].isin(total_sections)].reset_index(drop=True).copy()
for s in sources:
    tot[s] = tot[s].map(parse_total)
tot['ave_total'] = tot[sources].mean(axis=1)
tot = tot.dropna(subset=['ave_total'])

# last game listed = last matchup of the week; its over/under rows are a pair
away, home = tot['Team'].iloc[-2], tot['Team'].iloc[-1]
tiebreak = tot['ave_total'].iloc[-2:].mean()
print(f"Tiebreaker: {away} @ {home}, total = {tiebreak:.1f}")
tot[['Team'] + sources + ['ave_total']].iloc[-2:]

Tiebreaker: Broncos @ Chiefs, total = 43.1


,Team,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,ave_total
30,Broncos,43.5,43.5,42.5,43.0,43.5,42.5,43.5,43.0,42.5,43.055556
31,Chiefs,43.5,43.5,42.5,43.0,43.5,42.5,43.5,43.0,42.5,43.055556
